---
title: "Modelo LightGBM-Tweedie: Ruta Calle 72"
---

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import time

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from utils import eda_toolkit as toolkit
from utils import build_features as bf
from utils import trainer_toolkit as tt
from utils import test_toolkit as ts

from eval.scorer import DemandScorer
from eval.holdout import build_holdout_dataset, evaluate_on_holdout
from models.baselines import seasonal_naive, historical_median
from models.lgbm_tweedie_wrapper import LightGBMTweedieWrapper
from trainer.walk_forward import WalkForwardTrainer

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Parámetros del experimento
EXEC_ID = 1          # cambia el número para no sobrescribir una corrida anterior
N_TRIALS = 100       # usa 5 para una prueba rápida
H = 2                # horizonte: 2 franjas = 30 minutos
WINDOW_MAX = 26000   # ventana máxima de la búsqueda = calentamiento

# Núcleos para LightGBM: usa los núcleos físicos del computador
try:
    import psutil
    N_THREADS = psutil.cpu_count(logical=False) or os.cpu_count()
except ImportError:
    N_THREADS = max(1, (os.cpu_count() or 2) // 2)
print(f"Núcleos lógicos del computador: {os.cpu_count()} | núcleos que usará LightGBM: {N_THREADS}")

RESULTS_PATH = PROJECT_ROOT / "results" / "lgbm_cl72"
MODELS_PATH = PROJECT_ROOT / "models"
exec_dir = RESULTS_PATH / f"exec_{EXEC_ID}"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)
exec_dir.mkdir(parents=True, exist_ok=True)

# Partición 80/20 hecha en el EDA (03_exploratory_data_analysis_CL72)
train = pd.read_parquet(DATA_PROCESSED / "train_c72.parquet")
holdout = pd.read_parquet(DATA_PROCESSED / "holdout_c72.parquet")

feature_builder = lambda df: bf.build_features(df, dropna=True)

assert train.index.max() < holdout.index.min(), "Solapamiento temporal train/holdout"

print("=" * 70)
print(f"Train:   {len(train):>6} filas  ({train.index[0]} -> {train.index[-1]})")
print(f"Holdout: {len(holdout):>6} filas  ({holdout.index[0]} -> {holdout.index[-1]})")
print("=" * 70)

Núcleos lógicos del computador: 16 | núcleos que usará LightGBM: 10


Train:    56992 filas  (2023-01-01 04:30:00 -> 2025-05-26 17:15:00)
Holdout:  14248 filas  (2025-05-26 17:30:00 -> 2025-12-31 20:30:00)


# Introducción

El Gradient Boosting con LightGBM es un marco de aprendizaje supervisado que construye un ensamble aditivo de árboles de decisión mediante la optimización secuencial de una función de pérdida diferenciable. A diferencia de los modelos que imponen una ecuación paramétrica para la evolución de la media condicional, LightGBM aprende la relación entre las variables predictoras y la respuesta directamente a partir de los datos, sin asumir una forma funcional específica.

En el contexto de la demanda de pasajeros por franja horaria, esta arquitectura ofrece una alternativa a los modelos recurrentes y a los enfoques econométricos tradicionales, al permitir que la relación entre el calendario, la estacionalidad, los rezagos y la demanda futura sea aprendida directamente de los datos. La naturaleza de la respuesta, sin embargo, impone una restricción sobre la función de pérdida: la serie es un conteo de pasajeros cuya varianza crece con la media (ver la estimación de $p$ más adelante), estructura que una pérdida cuadrática o una verosimilitud Poisson no representan. Por esta razón se adopta la distribución Tweedie como familia de verosimilitud del objetivo, cuya forma compuesta Poisson-Gamma asigna masa puntual en cero y una densidad continua para los valores positivos.

El objetivo de esta sección es presentar, entrenar y evaluar modelos LightGBM con pérdida Tweedie para la predicción de la demanda de pasajeros por franja horaria en la ruta Calle 72, con el mismo protocolo aplicado en la ruta K54. Dado que la operación requiere pronósticos a múltiples escalas temporales, el problema se formula como una tarea de predicción directa multi-horizonte. Para cada intervalo operativo solicitado (30, 60 y 120 minutos) se entrena y optimiza un modelo LightGBM independiente, cada uno con su propio objetivo agregado y su propia validación. Este notebook desarrolla el intervalo de 30 minutos; los de 60 y 120 minutos se obtienen cambiando `H` a 4 y 8. La evaluación sobre el *holdout* replica las condiciones de despliegue operativo mediante un reentrenamiento rodante por bloques con ventana deslizante. Este esquema respeta la partición cronológica y las restricciones metodológicas del proyecto: emplea exclusivamente variables disponibles en el instante de decisión y previene cualquier fuga de información entre conjuntos.

### Arquitectura

LightGBM construye un modelo aditivo de árboles de decisión que aproxima la función objetivo de forma secuencial. Para un conjunto de $N$ observaciones $\{\mathbf{x}_i, y_i\}_{i=1}^{N}$, el modelo final se expresa como la suma de $M$ árboles $h_m(\cdot)$ escalados por una tasa de aprendizaje $\nu$:

$$
F_M(\mathbf{x}_i) = \sum_{m=1}^{M} \nu \, h_m(\mathbf{x}_i),
$$

donde cada árbol $h_m \in \mathcal{H}$ es una función escalonada sobre el espacio de predictores. El ajuste es *forward stagewise*: en la iteración $m$, el modelo previo $F_{m-1}$ se mantiene fijo y se añade un nuevo árbol que corrige sus errores,

$$
F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu \, h_m(\mathbf{x}).
$$

El nuevo árbol se obtiene minimizando una expansión de segundo orden de la pérdida $\mathcal{L}$ alrededor de la predicción actual. Definiendo el gradiente $g_{im}$ y el hessiano $h_{im}$ de la pérdida respecto a $F_{m-1}(\mathbf{x}_i)$,

$$
g_{im} = \left. \frac{\partial \mathcal{L}(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}, \qquad
h_{im} = \left. \frac{\partial^2 \mathcal{L}(y_i, F)}{\partial F^2} \right|_{F = F_{m-1}(\mathbf{x}_i)},
$$

las hojas del árbol se calculan con la solución de Newton,

$$
w_{j}^{*} = - \frac{\sum_{i \in I_j} g_{im}}{\sum_{i \in I_j} h_{im} + \lambda},
$$

donde $I_j$ es el conjunto de observaciones que caen en la hoja $j$ y $\lambda$ es el término de regularización $\ell_2$. La ganancia de una partición que divide $I$ en $I_L$ e $I_R$ se evalúa como:

$$
\mathcal{G} = \frac{1}{2}\left[
\frac{\left(\sum_{i \in I_L} g_{im}\right)^2}{\sum_{i \in I_L} h_{im} + \lambda} +
\frac{\left(\sum_{i \in I_R} g_{im}\right)^2}{\sum_{i \in I_R} h_{im} + \lambda} -
\frac{\left(\sum_{i \in I} g_{im}\right)^2}{\sum_{i \in I} h_{im} + \lambda}
\right].
$$

Bajo el objetivo `tweedie`, LightGBM asume que la respuesta sigue una distribución Tweedie con parámetro de potencia $p$ y utiliza una función de enlace logarítmica, de modo que la media condicional se obtiene como $\hat{\mu}(\mathbf{x}) = \exp(F(\mathbf{x}))$. La pérdida que se minimiza es la desviación negativa de la log-verosimilitud Tweedie,

$$
\mathcal{L}_{\text{Tweedie}}(y, \hat{\mu}) = - \left[ y \, \frac{\hat{\mu}^{1-p}}{1-p} - \frac{\hat{\mu}^{2-p}}{2-p} \right].
$$

:::{admonition} Información: Distribución Tweedie
:class: info

La distribución Tweedie con parámetro de potencia $p$ pertenece a la familia exponencial y se caracteriza por la relación entre la media $\mu$ y la varianza:

$$
\operatorname{Var}(y) = \phi \, \mu^{p}, \qquad 1 < p < 2.
$$

Valores de $p$ cercanos a 1 aproximan la verosimilitud a una Poisson; valores cercanos a 2 la aproximan a una Gamma; los valores intermedios corresponden a la familia compuesta Poisson-Gamma, que admite masa en cero y una componente continua positiva. En este proyecto, el parámetro $p$ se estima empíricamente a partir de la relación varianza-media de la serie. El valor resultante se fija como constante durante el entrenamiento para garantizar la estabilidad y comparabilidad de la métrica.
:::

Con la función de enlace logarítmica, las derivadas de la pérdida respecto a la predicción del modelo son:

$$
\frac{\partial \mathcal{L}}{\partial F} = -y \, e^{(1-p)F} + e^{(2-p)F}, \qquad
\frac{\partial^2 \mathcal{L}}{\partial F^2} = -y(1-p) \, e^{(1-p)F} + (2-p) \, e^{(2-p)F}.
$$

Estas expresiones alimentan el gradiente $g_{im}$ y el hessiano $h_{im}$ en cada iteración del boosting y son las que permiten que el modelo pondere de forma diferenciada las franjas con cero pasajeros y las franjas de alta demanda. Para $1 < p < 2$, el hessiano es estrictamente positivo, lo que garantiza la convexidad local del subproblema de optimización en cada hoja.

El modelo completo $F_M(\cdot)$ asigna a cada vector de predictores $\mathbf{x}$ (hora, día de la semana, mes, codificación cíclica, estacionalidad, rezagos y tendencia temporal) una predicción de la media condicional de pasajeros acumulados en la ventana objetivo, $\hat{\mu}(\mathbf{x}) = \exp(F_M(\mathbf{x}))$.

### Protocolo de entrenamiento y validación

Se entrena un modelo LightGBM independiente por cada horizonte de predicción (30, 60 y 120 minutos, equivalentes a $h \in \{2, 4, 8\}$ franjas). El *target* de cada modelo es la suma rodante de pasajeros desde el instante de decisión hasta el final del intervalo, compartiendo todos el mismo conjunto de 23 variables predictoras. Dado que la agregación temporal altera la dispersión de la demanda, el parámetro de potencia $p$ de Tweedie se estima empíricamente de forma independiente para cada horizonte y se fija como constante durante su entrenamiento. Cada intervalo se predice en un único paso, sin usar las predicciones de un horizonte como insumo de otro.

Para adaptar el modelo al nivel de demanda más reciente (en el análisis exploratorio, las subidas de un día hábil bajan un 13,1 % entre 2023 y 2025), se implementó una ventana deslizante (*sliding window*). El hiperparámetro `window` define la historia reciente que observa el modelo; se exploraron tamaños entre 18,000 y 26,000 franjas (de 8 a 12 meses), con un calentamiento inicial fijado en 26,000.

La validación en `train_c72.parquet` usa un esquema *walk-forward* de 5 pliegues, aplicando un *embargo* de $h-1$ franjas en la frontera para evitar fugas de información. Para la evaluación final, el `holdout_c72.parquet` replica esta mecánica avanzando cronológicamente y reentrenando el modelo por bloques con las últimas `window` franjas disponibles.

La implementación utiliza `LightGBM` (v4.6) con el objetivo `tweedie`. El *early stopping* interno se evalúa sobre el último tramo cronológico de cada bloque de entrenamiento (10 % en la búsqueda de hiperparámetros y 30 % en el modelo final), respetando también el *embargo* del horizonte.

## Ventana de predicción de 30 minutos

### Entrenamiento

Para determinar el parámetro de potencia $p$ en la muestra de entrenamiento, se aplicó un estimador por momentos basado en la relación teórica entre la varianza y la media de la distribución Tweedie:

$$
\log(\operatorname{Var}(y)) = \log(\phi) + p \log(\mu)
$$

Las observaciones del bloque de calentamiento (las primeras 26.000 franjas del entrenamiento) se agrupan por hora y día de la semana, y $p$ es la pendiente de la regresión de $\log(\operatorname{Var})$ sobre $\log(\mu)$ en esos grupos. Se usa la misma función que en la ruta K54 (`_estimate_tweedie_p_internal` de `trainer_toolkit`), y solo el bloque de calentamiento, para que el valor no use información de los bloques de validación. La gráfica compara la distribución observada del objetivo con la distribución Tweedie ajustada y su gráfico Q-Q.

In [2]:
warmup_data = train["pasajeros"].iloc[:WINDOW_MAX]
pasajeros_h2_warmup = warmup_data.rolling(window=H).sum().shift(-(H - 1)).dropna()

p_warmup = tt._estimate_tweedie_p_internal(pasajeros_h2_warmup)
TWEEDIE_P = round(p_warmup, 4)
print(f"Potencia Tweedie estimada con el calentamiento (h = {H}): p = {TWEEDIE_P}")
if TWEEDIE_P in (1.01, 1.99):
    print("Atención: p quedó en el límite del recorte (1.01–1.99); revisar antes de entrenar.")

_, _ = tt.plot_tweedie_fit(
    series=pasajeros_h2_warmup,
    p=p_warmup,
    titles=("Distribución de Pasajeros (Warmup)", "Q-Q Plot (Warmup)"),
    xlab=f"Pasajeros acumulados en 30 min (h={H})",
    pads=(15, 15, 12),
    legend_bbox=(1.0, 0.95)
)

Potencia Tweedie estimada con el calentamiento (h = 2): p = 1.1122


Con el bloque de calentamiento (las primeras 26.000 franjas, del 1 de enero de 2023 al 4 de febrero de 2024) se obtuvo $p = 1.1122$. El valor es menor que el de la ruta K54 ($p = 1.4311$): en Calle 72 la varianza crece más despacio con la media y la distribución está más cerca de una Poisson.

En la gráfica, la curva Tweedie sigue la forma del histograma en la zona central, entre unos 100 y 400 pasajeros cada 30 minutos. Las diferencias están en los extremos: los datos tienen más franjas con cero pasajeros de las que asigna la distribución ajustada (primera barra del histograma), y en el gráfico Q-Q los cuantiles observados quedan por encima de la recta a partir de unos 500 pasajeros, es decir, la cola derecha de los datos es más pesada que la de la distribución ajustada.

Para la selección de hiperparámetros del modelo LightGBM correspondiente al horizonte de 30 minutos ($h = 2$ franjas), se implementó una validación cruzada temporal de 5 pliegues sobre el conjunto de entrenamiento. La búsqueda de Optuna exploró la complejidad de los árboles de decisión, los parámetros de regularización y el tamaño de la ventana de historia reciente.

El modelo recibe directamente una matriz tabular con las 23 variables predictoras (calendario, codificación cíclica, estacionalidad, tendencia temporal y rezagos precalculados). Para evitar el sesgo de anticipación (*look-ahead bias*) y adaptarse al nivel de demanda más reciente, se implementó una estrategia de ventana deslizante (*sliding window*). En cada pliegue, el modelo se reentrena con un bloque de historia delimitado por el hiperparámetro `window` y se valida con el bloque cronológico inmediatamente posterior, aplicando un *embargo* de una franja ($h - 1 = 1$) para garantizar la integridad temporal de los datos.

El espacio de búsqueda incluyó el tamaño de la ventana de entrenamiento (`window`), el número máximo de hojas por árbol (`num_leaves`), la tasa de aprendizaje (`learning_rate`), el mínimo de datos y de suma de hessianos por hoja (`min_data_in_leaf`, `min_sum_hessian_in_leaf`), las fracciones y frecuencia de submuestreo (`bagging_fraction`, `bagging_freq`, `feature_fraction`) y las penalizaciones por regularización $\ell_1$ y $\ell_2$. El entrenamiento se configuró con un máximo de 5,000 árboles y una paciencia de 300 iteraciones para la parada temprana (*early stopping*) interna, la cual se evalúa sobre el último 10% cronológico de cada bloque de entrenamiento, respetando también el embargo de una franja. El desempeño se evaluó mediante la desviación Tweedie, fijando el parámetro de potencia en el valor estimado en la celda anterior (calculado solo con el bloque de calentamiento para evitar fuga de información) y aplicando un límite superior de 100 por observación para mitigar el efecto de eventos atípicos impredecibles.

In [3]:
def lgbm_search_space(trial: optuna.Trial) -> dict:
    return {
        "window": trial.suggest_categorical(
            "window", [18000, 20000, 22000, 24000, WINDOW_MAX]
        ),
        "num_leaves": trial.suggest_int("num_leaves", 15, 300, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 500, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0, step=0.05),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0, step=0.05),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 50.0, log=True),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_sum_hessian_in_leaf", 1e-3, 10.0, log=True
        ),
    }

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=H,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=1,
    n_trials=N_TRIALS,
    suggest_params_fn=lgbm_search_space,
    early_stop_rounds=30,
    inner_val_frac=0.1,
    model_kwargs={
        "tweedie_variance_power": TWEEDIE_P,
        "n_estimators": 5000,
        "early_stopping_rounds": 300,
        "num_threads": N_THREADS,
        "seed": 42,
    },
)

study = trainer.run(train)
best_params = dict(study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

La búsqueda realizó 81 de los 100 ensayos programados (numerados del 0 al 80). La mejor configuración se encontró en el ensayo 50 y los 30 siguientes no la mejoraron, lo que activó la parada temprana. El mecanismo de poda descartó 19 configuraciones antes de completar sus 5 pliegues y ninguna falló, por lo que 62 configuraciones completaron la validación. La menor desviación Tweedie promedio fue 15,026. El entrenamiento de los pliegues sumó 961,7 segundos (unos 16 minutos), con un promedio de 2,5 segundos por pliegue y LightGBM usando 10 núcleos.

In [4]:
resumen_data = []

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_tweedie = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(
            study_name=f"LightGBMTweedieWrapper_h{H}_study", 
            storage=storage_url
        )

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
        
        mejor_tweedie = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor Tweedie Dev": mejor_tweedie,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Fallidos,Mejor Tweedie Dev,Tiempo Total (s),Tiempo por Fold (s)
81,62,19,0,15.0256,961.7000,2.5310


Las cinco mejores configuraciones tienen una desviación Tweedie promedio entre 15,026 y 15,036, con una desviación estándar entre pliegues de 2,13 a 2,23. La diferencia entre ellas (0,01) es mucho menor que la variabilidad entre pliegues, por lo que en la práctica empatan. Las cinco eligieron la misma ventana de historia, 22.000 franjas (unos 338 días, poco más de 11 meses), y comparten una fracción de variables baja (0,40 a 0,45), una fracción de filas de 0,45 a 0,50 y penalizaciones $\ell_1$ y $\ell_2$ cercanas a cero (menores a $10^{-3}$). Varían en el número de hojas (29 a 69), el mínimo de datos por hoja (101 a 182) y la tasa de aprendizaje (0,007 a 0,022).

In [5]:
df_lgbm = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")

df_lgbm["lr"] = df_lgbm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lgbm["l1"] = df_lgbm["lambda_l1"].apply(lambda x: f"{x:.1e}")
df_lgbm["l2"] = df_lgbm["lambda_l2"].apply(lambda x: f"{x:.1e}")

cols_agrupar = [
    "window", 
    "num_leaves", 
    "min_data_in_leaf", 
    "feature_fraction", 
    "bagging_fraction", 
    "lr", 
    "l1",
    "l2"
]

df_agg = df_lgbm.groupby(cols_agrupar)["tweedie_deviance"].agg(["mean", "std"]).reset_index()

df_agg["Tweedie Deviance"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "window": "Ventana",
    "num_leaves": "Hojas",
    "min_data_in_leaf": "Min Data Hoja",
    "feature_fraction": "Feat. Frac",
    "bagging_fraction": "Bagging Frac",
    "lr": "LR",
    "l1": "L1",
    "l2": "L2"
})

df_final = df_top5[
    ["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac", "LR", "L1", "L2", "Tweedie Deviance"]
]

tt.display_styled_df(df=df_final, decimals=4, align_left_first=False)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"22,000",29,109,0.4000,0.4500,0.0221,9.0e-08,2.0e-05,15.026 ± 2.154
"22,000",50,128,0.4000,0.4500,0.0069,8.0e-07,8.4e-05,15.026 ± 2.132
"22,000",69,182,0.4000,0.5000,0.0193,2.8e-07,5.7e-04,15.030 ± 2.185
"22,000",46,104,0.4500,0.4500,0.0205,6.4e-08,1.4e-05,15.033 ± 2.190
"22,000",42,101,0.4000,0.4500,0.0153,2.9e-08,6.0e-04,15.036 ± 2.229


El modelo final usa la configuración con menor desviación promedio (`study.best_params`, primera fila de la tabla): ventana de 22.000 franjas, 29 hojas, tasa de aprendizaje de 0,022, mínimo de 109 datos por hoja, fracción de variables de 0,40 y fracción de filas de 0,45. La segunda configuración tiene el mismo promedio con tres decimales y una variabilidad entre pliegues ligeramente menor (2,132 frente a 2,154); la diferencia entre ambas no es relevante. El modelo se entrena una sola vez con las últimas 22.000 franjas del conjunto de entrenamiento, del 22 de junio de 2024 al 26 de mayo de 2025.

In [6]:
best_params = dict(study.best_params)
for k, val in best_params.items():
    print(f"{k:>24}: {val}")

window = best_params["window"]
model_params = {k: v for k, v in best_params.items() if k != "window"}

final_kwargs = {
    "tweedie_variance_power": TWEEDIE_P,
    "n_estimators": 5000,
    "early_stopping_rounds": 300,
    "inner_val_frac": 0.3,
    "num_threads": N_THREADS,
    "seed": 42,
}

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=H,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=1,
    n_trials=N_TRIALS,
    model_kwargs={},
)

X_train_full, y_train_full, _, _ = trainer._build_dataset(train)

model_production = LightGBMTweedieWrapper(**model_params, **final_kwargs)
model_production.fit(X_train_full.iloc[-window:], y_train_full[-window:])

                  window: 22000
              num_leaves: 29
           learning_rate: 0.02214065095014879
        min_data_in_leaf: 109
        feature_fraction: 0.4
        bagging_fraction: 0.45
            bagging_freq: 2
               lambda_l1: 9.028957269769347e-08
               lambda_l2: 2.046708015812977e-05
 min_sum_hessian_in_leaf: 1.1365415517056803


Para evaluar la estructura de los residuos tanto en el conjunto de entrenamiento (*In-Sample*) como en el de prueba (*Out-of-Sample*), a partir de las demandas predichas y las observaciones reales, se calcularon los residuos y su versión estandarizada mediante las siguientes expresiones:

$$
e_t = y_t - \hat{y}_t, \qquad z_t = \frac{e_t - \bar{e}}{s_e}
$$

donde $y_t$ es la demanda observada, $\hat{y}_t$ es la predicción del modelo, $\bar{e}$ es la media muestral de los residuos y $s_e$ es su desviación estándar.

Debido a que los estadísticos de Ljung-Box y ARCH-LM escalan linealmente con el tamaño de la muestra, evaluar la totalidad de la ventana activa de entrenamiento (`window` franjas) introduce un efecto de sobre-potencia (*over-power*) que infla los estadísticos ante cualquier dependencia residual. Para atenuar este sesgo sin alterar la secuencia cronológica de los rezagos, el diagnóstico se realizó sobre el último bloque continuo de entrenamiento ($N = 1000$ franjas). Sobre ese bloque se aplican las pruebas de Ljung-Box (autocorrelación) y ARCH-LM (heterocedasticidad).

En Calle 72 ambas pruebas rechazan sus hipótesis nulas en los tres rezagos evaluados ($p < .001$): el estadístico de Ljung-Box está entre 180,3 y 192,2 y el de ARCH-LM entre 88,8 y 102,6.

In [7]:
preds_train_final = model_production.predict_in_sample(X_train_full[-window:])
residuals_train = y_train_full[-window:] - preds_train_final

block_size = 1000
residuals_block = residuals_train[-block_size:]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_train = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_train,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),180.332,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),182.565,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),192.179,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),88.811,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),92.467,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),102.568,< .001,< .001,Efectos ARCH Significativos


La autocorrelación de los residuos es consistente con que el modelo no usa rezagos menores a 32 franjas (8 horas) y con que el objetivo de 30 minutos se superpone entre franjas consecutivas. Los efectos ARCH indican que la dispersión del error cambia en el tiempo, lo esperable cuando la varianza de la demanda crece con la media.

La gráfica compara la demanda observada con la predicha en la ventana de entrenamiento. El panel superior suma por día el objetivo de 30 minutos; como cada franja aparece en dos ventanas consecutivas, esos totales equivalen aproximadamente al doble de los pasajeros del día. Los paneles inferiores muestran por franja la semana del 2 al 8 de febrero de 2025 y el 28 de febrero de 2025.

In [8]:
ts.plot_temporal_fit_panel(
    dates=X_train_full.index[-window:],
    y_true=y_train_full[-window:],
    y_pred=preds_train_final,
    zoom_left=("2025-02-02", "2025-02-08"),
    zoom_right=("2025-02-28", "2025-02-28"),
    text_left="Febrero 2025",
    text_right="28 Febrero 2025",
    title="Demanda de pasajeros 2024-2025",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros por día",
    ylab_zoom="Pasajeros por franja",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1, wspace = 0.05
)

El modelo sigue el ciclo semanal en toda la ventana de entrenamiento: niveles altos en días hábiles, intermedios los sábados y bajos los domingos. Las mayores diferencias se ven en días puntuales con caídas fuertes, como el 1 de enero de 2025, donde la demanda observada queda por debajo de la predicha. En los acercamientos por franja, el modelo reproduce la forma del día pero suaviza los picos: el 28 de febrero de 2025 predice cerca de 490 pasajeros en el pico de la mañana frente a unos 650 observados, y no reproduce el pico de las 13:00 (unos 550 observados).

## Evaluación en el holdout

La evaluación del rendimiento del modelo LightGBM-Tweedie requiere contrastar su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre demanda no vista. Mientras que la ventana activa de entrenamiento (*In-Sample*, `window` franjas) permite verificar la convergencia del algoritmo a la dinámica histórica de operación, la evaluación fuera de muestra (*Out-of-Sample*, conjunto *holdout*) determina la utilidad operativa real del modelo ante nuevos regímenes de movilidad.

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se aplicaron los criterios establecidos en el protocolo experimental. Estos criterios se basan en el error relativo porcentual entre las métricas de prueba y entrenamiento, denotado como:

$$
\Delta_{\text{rel}} (\%) = \frac{M_{\text{test}} - M_{\text{train}}}{|M_{\text{train}}|} \times 100
$$

Se estableció un umbral de tolerancia relativo de $|\Delta_{\text{rel}}| = 5\%$ sobre la función de pérdida principal (*Tweedie Deviance*, con la potencia estimada). Un incremento relativo superior a este margen evidenciaría degradación en la capacidad de generalización.

No obstante, en el contexto del pronóstico de demanda de pasajeros bajo verosimilitud Tweedie, debe considerarse la naturaleza asimétrica de esta función de pérdida y su sensibilidad diferenciada entre franjas de baja afluencia (incluyendo cierres operativos en cero) y horas pico. Dado que un desplazamiento puntual en la demanda real afecta de forma no lineal las métricas agregadas, la lectura de $\Delta_{\text{rel}}$ se complementó con métricas de escala (`MAE`, `RMSE`), métricas relativas (`SMAPE`, `MASE`), el diagnóstico de sesgo (`Bias`) y la inspección visual de las trayectorias diarias e intradía.

Las predicciones fuera de muestra se obtuvieron ejecutando el método `forecast` de `LightGBMTweedieWrapper` sobre la serie continua concatenada (garantizando la disponibilidad exacta de los rezagos operativos sin pérdida de observaciones iniciales ni fuga de información). Este procedimiento aplicó un esquema de reentrenamiento periódico por bloques (`n_splits = 5`) con ventana deslizante fija (`window` elegido en la búsqueda, `embargo = 1`), generando pronósticos puntuales en la escala original de pasajeros por franja de 15 minutos.

In [9]:
df_eval_context = pd.concat([train, holdout], axis=0)
X_eval_all, y_eval_all, y_naive_eval_all, _ = trainer._build_dataset(df_eval_context)

holdout_mask = X_eval_all.index >= holdout.index.min()
X_holdout_final = X_eval_all.loc[holdout_mask]
y_holdout_final = y_eval_all[holdout_mask]
y_naive_holdout_final = y_naive_eval_all[holdout_mask]

start_time = time.time()
preds_holdout = model_production.forecast(
    X_seq=X_holdout_final,
    y_seq=y_holdout_final,
    window=window,
    embargo=trainer.embargo,
    n_splits=trainer.n_splits,
)
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": X_holdout_final.index,
    "y_true": y_holdout_final,
    "y_naive": y_naive_holdout_final,
    "y_pred": preds_holdout,
})

output_path = RESULTS_PATH / "lgbm_holdout_predictions.csv"
df_preds.to_csv(output_path, index=False)

Diagnóstico de los residuos fuera de muestra: pruebas de Ljung-Box y ARCH-LM sobre las últimas 1.000 franjas del *holdout*. Ambas rechazan sus hipótesis nulas en los tres rezagos ($p < .001$), con estadísticos mayores que en entrenamiento: Ljung-Box entre 656,2 y 863,5 y ARCH-LM entre 280,0 y 300,8. El error fuera de muestra conserva estructura temporal y su dispersión cambia en el tiempo.

In [10]:
residuals_test = (
    df_preds["y_true"].to_numpy() - df_preds["y_pred"].to_numpy()
)

block_size = 1000
residuals_block = residuals_test[-block_size:]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_test = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_test,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),656.192,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),764.879,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),863.451,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),280.010,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),292.780,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),300.777,< .001,< .001,Efectos ARCH Significativos


Comparación de métricas entre la ventana de entrenamiento y el *holdout*, junto con el modelo ingenuo semanal (misma franja de la semana anterior). Un aumento relativo mayor al 5 % en la desviación Tweedie indicaría pérdida de capacidad de generalización.

La desviación Tweedie baja un 0,68 % del entrenamiento al *holdout* (de 11,976 a 11,895), dentro de la tolerancia, por lo que no hay evidencia de sobreajuste. El MAE (50,8 y 50,3), el RMSE (67,1 y 67,0) y el SMAPE (0,284 y 0,287) prácticamente no cambian. El MASE sube un 7,5 % (de 0,637 a 0,685), pero sigue por debajo de 1 en ambos periodos, es decir, el modelo se equivoca menos que el modelo ingenuo: en el *holdout*, su MAE es de 50,3 pasajeros por cada 30 minutos frente a 73,4 del ingenuo (31,5 % menos), con una demanda media de 206 pasajeros. El sesgo pasa de −1,1 en entrenamiento a +11,0 en el *holdout*: fuera de muestra el modelo sobreestima en promedio 11 pasajeros por cada 30 minutos (5 % de la demanda media). El error relativo de +1.090 % del sesgo se debe a que el valor de entrenamiento es cercano a cero.

In [11]:
df_preds = pd.read_csv(RESULTS_PATH / "lgbm_holdout_predictions.csv")

X_train_full, y_train_full, y_naive_train_full, _ = trainer._build_dataset(train)
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])

ts.display_train_test_comparison(
    y_train=y_train_full[-window:],
    preds_train=preds_train_final,
    y_naive_train=y_naive_train_full[-window:],
    y_test=df_preds["y_true"].to_numpy(),
    preds_test=df_preds["y_pred"].to_numpy(),
    y_naive_test=df_preds["y_naive"].to_numpy(),
    p=TWEEDIE_P, error_relativo=True
)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,11.9760,11.8947,-0.6791%
MAE,50.8383,50.2733,-1.1115%
RMSE,67.1367,67.0174,-0.1778%
MSE,4507.3387,4491.3286,-0.3552%
Bias,-1.1123,11.0127,+1090.1159%
SMAPE,0.2837,0.2872,+1.2359%
MASE,0.6374,0.6852,+7.5130%
Pinball (τ=0.50),25.4192,25.1366,-1.1115%


Ajuste en el *holdout*: demanda observada frente a predicha, por día y en dos acercamientos por franja. Como en la ventana de entrenamiento, los totales diarios del panel superior equivalen aproximadamente al doble de los pasajeros del día. El primer punto es bajo porque el *holdout* empieza el 26 de mayo de 2025 a las 17:30 y ese día solo tiene 13 franjas.

La sobreestimación no es pareja en el periodo: el sesgo medio es del 10 % en junio, 9 % en julio y 7 % en agosto, baja al 5 % en septiembre, al 1,5 % en octubre y al 0,1 % en noviembre, y sube al 5 % en diciembre. Los días con mayor sobreestimación son días con demanda atípicamente baja: el 19 de junio, el 9 y el 29 de julio y el 24 de diciembre (señalados como atípicos en el análisis exploratorio), el 25 de diciembre y el 31 de diciembre de 2025; en ellos la predicción supera a lo observado entre un 38 % y un 91 %. En los acercamientos por franja, el modelo sigue la forma de los días hábiles de la primera semana de septiembre y, como en entrenamiento, suaviza los picos: el 15 de octubre predice cerca de 520 pasajeros en el pico de la mañana frente a unos 790 observados.

In [12]:
ts.plot_temporal_fit_panel(
    dates=pd.to_datetime(df_preds["date"]),
    y_true=df_preds["y_true"].to_numpy(),
    y_pred=df_preds["y_pred"].to_numpy(),
    zoom_left=("2025-09-01", "2025-09-07"),
    zoom_right=("2025-10-15", "2025-10-15"),
    text_left="Septiembre 2025",
    text_right="15 Octubre 2025",
    title="Demanda de pasajeros Holdout 2025 (Out-of-Sample)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros por día",
    ylab_zoom="Pasajeros por franja",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1,
    wspace=0.05,
)